# Lab #2

The purpose of this laboratory is to put NumPy, pandas and Matplotlib to work on two datasets.
More specifically, you will learn how to:
- load and explore a dataset with pandas, and look at it with Matplotlib;
- move from a pandas DataFrame to NumPy arrays;
- implement your first Machine Learning model, the K-Nearest Neighbors (KNN) classifier, with NumPy, and measure its accuracy;
- (optionally) extend it with inheritance, the way scikit-learn models share one interface;
- explore a larger real-world dataset, the Points of Interest of New York City, with pandas and Matplotlib.

## 0 Setup
### 0.1 Kernel and libraries
This is the first lab you run on your own machine. Open this notebook in VS Code and select, with the kernel picker at the top right, the virtual environment you created for the course (`.venv`).

Then run the cell below: it should print the three versions without errors. If `import numpy` fails, the notebook is almost certainly running on the wrong kernel.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("NumPy", np.__version__)
print("pandas", pd.__version__)
print("Matplotlib", matplotlib.__version__)

### 0.2 The datasets
In this laboratory you will use two datasets:
- **Iris**, which you already met in Lab 1: 150 flowers, each described by 4 measurements in cm (sepal length, sepal width, petal length, petal width) and by its species (Iris-setosa, Iris-versicolor or Iris-virginica). It is a CSV file without a header, available [here](https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data).
- **New York Points of Interest (POIs)**, a [zip file](https://github.com/dbdmg/data-science-lab/raw/master/datasets/NYC_POIs.zip) with three files:
    - `pois_all_info`: the list of the POIs of New York City, in TSV (Tab-Separated Values) format, i.e. a CSV file whose fields are separated by a TAB character instead of a comma;
    - `ny_municipality_pois_id.csv`: the `@id` of the POIs inside the New York City municipality, one per line;
    - `New_York_City_Map.PNG`: a map of the New York City municipality.

The fields of `pois_all_info` are:
- **@id**: the identifier of the POI;
- **@type**: the kind of OpenStreetMap element (`node`, `way` or `relation`);
- **@lat**, **@lon**: the latitude and longitude of the POI, in decimal degrees;
- **amenity**, **shop**, **public_transport**, **highway**: the four POI categories. When a POI belongs to a category, the corresponding field holds its type (e.g. `amenity` = `restaurant`), otherwise it is empty;
- **name**: the name of the POI, if any.

The cell below downloads both datasets in the folder of this notebook, unless they are already there.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

IRIS_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
POIS_URL = "https://github.com/dbdmg/data-science-lab/raw/master/datasets/NYC_POIs.zip"

if not Path("iris.csv").exists():
    urllib.request.urlretrieve(IRIS_URL, "iris.csv")

if not Path("pois_all_info").exists():
    urllib.request.urlretrieve(POIS_URL, "NYC_POIs.zip")
    with zipfile.ZipFile("NYC_POIs.zip") as archive:
        for name in archive.namelist():
            if not name.startswith("__MACOSX"):  # skip the macOS metadata stored in the archive
                archive.extract(name)

for name in ["iris.csv", "pois_all_info", "ny_municipality_pois_id.csv", "New_York_City_Map.PNG"]:
    print(f"{name}: {Path(name).stat().st_size} bytes")

## Exercises
Note that exercises marked with a (*) are optional, you should focus on completing the other ones first.

## 1 Iris: from exploration to a classifier
In Lab 1 you loaded Iris as a list of lists and computed its statistics with loops. Here you do the same with pandas, look at the data with Matplotlib, and then build a classifier with NumPy.

### 1.1 Load the dataset
Load `iris.csv` into a DataFrame `iris_df` whose columns are named `sepal_length`, `sepal_width`, `petal_length`, `petal_width` and `class` (see the `header` and `names` parameters of `pd.read_csv`).

*Check*: the shape of `iris_df` is `(150, 5)`, and its first row is `5.1, 3.5, 1.4, 0.2, Iris-setosa`.

### 1.2 Statistics per species
Compute the mean and the standard deviation of the 4 measurements for each species, with a single pandas expression.

Compare the result with Exercise 3 of Lab 1: the means are the same, the standard deviations are not. Why? How do you get the values of Lab 1 back?

TODO: write your comment here

### 1.3 Distribution of the features
In one figure, draw a box plot of each of the 4 features, with the feature names on the x axis (`tick_labels` parameter of `boxplot`) and the unit on the y axis.

### 1.4 Distribution of the features per species
Draw the box plots of Exercise 1.3 for each species separately, as three plots side by side in the same figure, each with the species as title. The three plots must use the same y scale, otherwise they cannot be compared (see the `sharey` parameter of `plt.subplots`).

Which features would you use to tell the three species apart?

TODO: write your comment here

### 1.5 Pairs of features
A box plot shows one feature at a time. Now look at two features at a time: in one figure, draw a scatter plot for every pair of features (how many pairs are there?), with each species in its own color, a legend, and the feature names on the axes. `itertools.combinations` generates the pairs.

Does the figure confirm your answer to Exercise 1.4?

TODO: write your comment here

### 1.6 From pandas to NumPy
The classifier you are going to write works on NumPy arrays. Build:
- `X`, the 4 measurements, an array of shape `(150, 4)`;
- `y`, the species as integers, an array of shape `(150,)`, with this mapping:
    - Iris-setosa: 0
    - Iris-versicolor: 1
    - Iris-virginica: 2

*Check*: `X` has a float `dtype`, `y` an integer one, and `y` contains 50 zeros, 50 ones and 50 twos.

### The K-Nearest Neighbors algorithm
In the rest of this part you will implement your own version of the K-Nearest Neighbors (KNN) algorithm, and use it to assign a species (i.e. a label) to flowers whose species is unknown.

The KNN algorithm is straightforward. Suppose that the measurements (e.g. the Iris features) and the label (e.g. the Iris species) of a set of samples are known in advance.

<img src="https://mlarchive.com/wp-content/uploads/2022/09/img2.png" width="800">

Then, whenever we want to label a new sample, we look at the K most similar samples (a.k.a. neighbors) and assign a label accordingly.

<img src="https://mlarchive.com/wp-content/uploads/2022/09/img1-1.png" width="800">

The simplest solution is a majority voting scheme: the new sample gets the label shared by most of its neighbors.
This approach is naive only at first sight: as the plots of Exercise 1.5 show, flowers of the same species do have similar measurements.
KNN is rarely the best model, but it is a valid baseline for many tasks.

### 1.7 Nearest neighbors of a sample
Define a function `k_nearest_neighbors(x, X, k)` that returns the indices of the `k` rows of `X` closest to the sample `x`, using the euclidean distance. Do not loop over the rows of `X`. (`np.linalg.norm` computes the norm of vectors.)

Apply it to the 36th sample (`X[35]`) with `k=5`, and print the indices and the labels of its neighbors.
- Which species would you guess for this sample?
- One of the neighbors is a bit special: which one, and why?

TODO: write your comment here

### 1.8 Training set and test set
To measure how good a classifier is, it has to be tested on samples it has never seen. Split the dataset at random:
- 20% of the samples form the **test set**: their features in `X_test`, their labels in `y_test`;
- the remaining 80% form the **training set**, `X_train` and `y_train`: this is the knowledge the model will use to make its predictions.

Each sample must end up in exactly one of the two sets, and the split must be the same at every run. (`np.random.default_rng(seed)` creates a random generator with a fixed seed; look at its `permutation` method.)

*Check*: the shapes are `(30, 4)` and `(30,)` for the test set, `(120, 4)` and `(120,)` for the training set.

Why would the first 30 rows of the file make a bad test set?

TODO: write your comment here

### 1.9 KNN as a class
From the next labs you will use the `scikit-learn` library, whose models share the same object-oriented interface: a `fit` method that learns from the training data and a `predict` method that labels new samples. Implement KNN with this interface, starting from the skeleton below (you are free to add other methods).
- `__init__` stores the number of neighbors `k`.
- `fit` stores the training data: KNN does not really learn anything, the model *is* the training set.
- `predict` labels each row of `X` with the label that is most frequent among its `k` nearest neighbors in the training set (majority voting). Reuse `k_nearest_neighbors`; `np.unique` can count how many times each value appears (see its `return_counts` parameter).
- (*) Compute the distances between all the rows of `X` and all the training samples at once, without looping over `X`. What shape must the array of differences have?

In [ ]:
class KNearestNeighbors:
    def __init__(self, k):
        """
        Store the value of k in an attribute of the class and initialize other attributes.
        :param k: int, number of neighbors to consider.
        """
        pass  # TODO: implement it!

    def fit(self, X, y):
        """
        Store the 'prior knowledge' of your model, that will be used to predict new labels.
        :param X: input data points, ndarray, shape = (R, C).
        :param y: input labels, ndarray, shape = (R,).
        """
        pass  # TODO: implement it!

    def predict(self, X):
        """
        Run the KNN classification on X.
        :param X: input data points, ndarray, shape = (N, C).
        :return: labels, ndarray, shape = (N,).
        """
        pass  # TODO: implement it!

### 1.10 Accuracy
Create a model with `k=3`, fit it on the training set, and predict the species of the test set into an array `y_pred`. Then compute the **accuracy** of the model, the fraction of test samples whose species has been guessed correctly:

$$ \text{accuracy} = {\text{number of correct predictions} \over \text{total number of predictions}} $$

*Check*: on Iris, the accuracy of KNN is usually 0.9 or more (guessing at random would give about 0.33).

### 1.11 Where are the mistakes?
The accuracy is a single number; a plot shows where the mistakes are. Using the two petal features, draw in one figure:
- the training samples, colored by species;
- the test samples on top of them, with a different marker and colored by *predicted* species;
- the misclassified test samples, highlighted so that they stand out (e.g. circled: see the `facecolors` and `edgecolors` parameters of `scatter`).

Where do the misclassified samples lie?

TODO: write your comment here

### Inheritance in a nutshell
A class can be defined as an extension of another one: `class Child(Parent)` **inherits** all the attributes and methods of `Parent`, and can add new ones or **override** (redefine) existing ones. Run the example below.
- `Square` and `Circle` inherit `describe` from `Shape`, and override `area`.
- `super()` gives access to the methods of the parent class, even those the child has overridden. It has two typical uses:
    - in the constructor, `super().__init__(...)` runs the constructor of the parent, so that the attributes it sets (here `self.name`) exist in the child too. A child that defines its own `__init__` *replaces* the one of the parent: without this call, those attributes are never set;
    - in any other method, `super().method(...)` *extends* the version of the parent instead of rewriting it: `Square.describe` calls `Shape.describe` and adds the side.
- `describe` is written once, in `Shape`, but calls `self.area()`: Python runs the `area` of the class the object actually belongs to. The loop treats the three objects in the same way, and each behaves in its own way: this is called **polymorphism**.

*Note*: a method whose name starts with two underscores (e.g. `__area`) is renamed by Python to `_Shape__area` inside `Shape` and to `_Square__area` inside `Square`, so a child class cannot override it. Methods meant to be overridden must not start with `__`, unless they also end with `__`, like `__init__`: these special methods are never renamed.

In [ ]:
import math


class Shape:
    def __init__(self, name):
        self.name = name

    def area(self):
        return 0.0

    def describe(self):
        return f"{self.name} with area {self.area():.2f}"


class Square(Shape):
    def __init__(self, side):
        super().__init__("square")  # runs Shape.__init__, which sets self.name
        self.side = side

    def area(self):  # overrides Shape.area
        return self.side**2

    def describe(self):  # extends Shape.describe
        return super().describe() + f" (side {self.side})"


class Circle(Shape):
    def __init__(self, radius):
        super().__init__("circle")
        self.radius = radius

    def area(self):
        return math.pi * self.radius**2


for shape in [Shape("point"), Square(3), Circle(1)]:
    print(shape.describe())

### 1.12 (*) Other distances
Your KNN uses the euclidean distance, but other distances may work better on other data, e.g.:
- the Minkowski distance of order $p$: $ minkowski_p(a, b) = \left(\sum_{i=1}^n |a_i - b_i|^p\right)^{1/p} $, which is the Manhattan distance for $p = 1$ and the euclidean one for $p = 2$;
- the cosine distance: $ cosine(a, b) = 1 - \frac{\sum_{i=1}^n a_i b_i}{ \sqrt{\sum_{i=1}^n a_i^2} \cdot \sqrt{\sum_{i=1}^n b_i^2}} $

Add them through inheritance, without copying the KNN algorithm:
1. Rewrite `KNearestNeighbors` so that the distance is computed by a method of its own, `distances(self, x)`, which returns the euclidean distances between `x` and each training sample, and that `predict` uses.
2. Define `MinkowskiKNN(k, p)`, which inherits from `KNearestNeighbors`, takes the order `p` as an extra constructor parameter, and overrides `distances`. Its constructor must still store `k` and initialize the training data: do it without repeating the code of the parent.
3. Define `CosineKNN(k)`, which overrides `distances`. The norms of the training samples do not change from one prediction to the next: compute them once, in `fit`, by extending the `fit` of the parent rather than rewriting it.
4. With a single loop over `KNearestNeighbors(k=3)`, `MinkowskiKNN(k=3, p=1)`, `MinkowskiKNN(k=3, p=3)` and `CosineKNN(k=3)`, fit each model on the training set and print its accuracy on the test set.

What happens if `MinkowskiKNN.__init__` does not call `super().__init__`? Which part of the code is polymorphic?

TODO: write your comment here

#

---
## 2 New York Points of Interest
Iris is small and clean. The POIs dataset is larger, has many empty fields and needs some work before it can be analyzed: a typical situation with real-world data.

### 2.1 Load the dataset
Load `pois_all_info` into a DataFrame `all_pois_df`, with the `@id` column as index (see the `index_col` parameter of `pd.read_csv`). Remember that it is a TSV file.

*Check*: `all_pois_df` has 77884 rows.

pandas probably warns you that some columns have mixed types. Look at the data: why does it happen? Find in the documentation of `pd.read_csv` how to avoid it.

TODO: write your comment here

### 2.2 POIs on the map
Write a function `draw_map(ax)` that draws the map of New York City on the axes `ax`: load the image with `plt.imread("New_York_City_Map.PNG")` and show it with `ax.imshow(img, extent=[-74.258, -73.7, 40.49, 40.92])`. The `extent` maps the pixels of the image to longitude (x) and latitude (y), so that the POIs land in the right place. A few POIs lie outside the map: keep the axes within its borders.

Then draw the map and all the POIs on it, small enough not to hide the map.

### 2.3 POIs of the New York City municipality
Many POIs lie outside the municipality. Keep in a DataFrame `pois_df` only the POIs whose `@id` is listed in `ny_municipality_pois_id.csv` (a file without a header), and plot them on the map. (`Index.isin` may help.)

*Check*: `pois_df` has 53521 rows. If you get more, find out why.

### 2.4 A category column
Add to `pois_df` a column `category` holding the category of each POI (`amenity`, `shop`, `public_transport` or `highway`), i.e. the name of the category field that is not empty; leave it empty for the POIs without a category. Then print how many POIs fall in each category, the empty one included (see `value_counts` and its `dropna` parameter).

How many POIs have more than one category? Which category does your code assign them, and why?

TODO: write your comment here

### 2.5 POIs per category
Draw the number of POIs of each category as a bar chart, with a title and a label on the y axis.

### 2.6 Most frequent amenity types
Show the 10 most frequent amenity types (the values of the `amenity` column), with their number of POIs, from the most to the least frequent.

### 2.7 The top 5 amenity types on the map
Plot on the map the POIs of the 5 most frequent amenity types, each type in its own color, with a legend. (The points are small: see the `markerscale` parameter of `legend`.)

### 2.8 (*) The top 5 types of each category
For each of the 4 categories, draw a bar chart of the 5 most frequent types among the POIs of that category, as assigned in Exercise 2.4 (e.g. the 5 most frequent `shop` types among the `shop` POIs), as four plots side by side in one figure, each with the category as title. The type names are long: keep them from overlapping (see `tick_params`).

How many bars does `highway` have with the order you chose in Exercise 2.4, and why?

TODO: write your comment here